In [2]:
print("code")

code


In [3]:
imgpath = "/home/azureuser/kaithi/OCD/odata/image.png"

In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    interact,
    IntSlider,
    FloatSlider,
    Checkbox,
    Dropdown,
)



SOURCE_BGR = cv2.imread(imgpath)
if SOURCE_BGR is None:
    raise FileNotFoundError(f"Could not read image: {imgpath}")

CURRENT_RESULTS = {}
CURRENT_PARAMS = {}


def odd(x: int) -> int:
    x = int(x)
    return x if x % 2 == 1 else x + 1


In [10]:
def preprocess_for_detection(
    source_bgr,
    resize_factor=1.0,

    # grayscale / normalization
    do_bg_normalization=True,
    bg_blur_ksize=81,

    # contrast
    do_clahe=True,
    clahe_clip=2.0,
    clahe_grid=8,

    # denoise / sharpen
    denoise_h=6,
    median_ksize=3,
    sharpen_amount=0.3,
    sharpen_sigma=1.0,

    # thresholding
    threshold_method="adaptive_gaussian",   # adaptive_gaussian / adaptive_mean / otsu / none
    block_size=41,
    c_value=9,
    invert_binary=False,

    # morphology
    morph_open=0,
    morph_close=3,
    dilate_iter=0,

    # bbox filtering
    min_area=30,
    max_area=50000,
    min_width=3,
    min_height=3,
    max_width_ratio=0.95,
    max_height_ratio=0.95,
):
    img = source_bgr.copy()

    if resize_factor != 1.0:
        new_w = max(1, int(img.shape[1] * resize_factor))
        new_h = max(1, int(img.shape[0] * resize_factor))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    work = gray.copy()

    # 1) background normalization
    bg_norm = work.copy()
    if do_bg_normalization:
        k = odd(bg_blur_ksize)
        bg = cv2.GaussianBlur(work, (k, k), 0)
        bg_norm = cv2.divide(work, bg, scale=255)
        work = bg_norm.copy()

    # 2) local contrast
    enhanced = work.copy()
    if do_clahe:
        clahe = cv2.createCLAHE(
            clipLimit=float(clahe_clip),
            tileGridSize=(int(clahe_grid), int(clahe_grid)),
        )
        enhanced = clahe.apply(work)
        work = enhanced.copy()

    # 3) denoise
    denoised = work.copy()
    if denoise_h > 0:
        denoised = cv2.fastNlMeansDenoising(
            work, None, h=int(denoise_h), templateWindowSize=7, searchWindowSize=21
        )
        work = denoised.copy()

    # 4) median blur
    medianed = work.copy()
    if median_ksize >= 3:
        mk = odd(median_ksize)
        medianed = cv2.medianBlur(work, mk)
        work = medianed.copy()

    # 5) mild sharpen
    sharpened = work.copy()
    if sharpen_amount > 0:
        blur = cv2.GaussianBlur(work, (0, 0), sharpen_sigma)
        sharpened = cv2.addWeighted(work, 1.0 + sharpen_amount, blur, -sharpen_amount, 0)
        work = sharpened.copy()

    # 6) threshold
    if threshold_method == "none":
        binary = work.copy()
    elif threshold_method == "adaptive_gaussian":
        binary = cv2.adaptiveThreshold(
            work, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            odd(block_size),
            int(c_value),
        )
    elif threshold_method == "adaptive_mean":
        binary = cv2.adaptiveThreshold(
            work, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            odd(block_size),
            int(c_value),
        )
    elif threshold_method == "otsu":
        _, binary = cv2.threshold(work, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        raise ValueError("Unknown threshold_method")

    if threshold_method != "none" and invert_binary:
        binary = 255 - binary

    # 7) morphology
    processed = binary.copy()

    if threshold_method != "none":
        if morph_open > 0:
            ok = odd(morph_open)
            kernel = np.ones((ok, ok), np.uint8)
            processed = cv2.morphologyEx(processed, cv2.MORPH_OPEN, kernel)

        if morph_close > 0:
            ck = odd(morph_close)
            kernel = np.ones((ck, ck), np.uint8)
            processed = cv2.morphologyEx(processed, cv2.MORPH_CLOSE, kernel)

        if dilate_iter > 0:
            kernel = np.ones((3, 3), np.uint8)
            processed = cv2.dilate(processed, kernel, iterations=int(dilate_iter))

    # 8) contour source
    if threshold_method == "none":
        _, contour_src = cv2.threshold(work, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        contour_src = 255 - processed if np.mean(processed) > 127 else processed.copy()

    # 9) contours to boxes
    contours, _ = cv2.findContours(contour_src, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h_img, w_img = contour_src.shape[:2]
    max_w = int(w_img * max_width_ratio)
    max_h = int(h_img * max_height_ratio)

    boxes = []
    overlay = rgb.copy()

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h

        if area < min_area or area > max_area:
            continue
        if w < min_width or h < min_height:
            continue
        if w > max_w or h > max_h:
            continue

        boxes.append((x, y, w, h))
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (255, 0, 0), 2)

    return {
        "original_rgb": rgb,
        "gray": gray,
        "bg_norm": bg_norm,
        "enhanced": enhanced,
        "denoised": denoised,
        "sharpened": sharpened,
        "binary": binary,
        "processed": processed,
        "contour_src": contour_src,
        "overlay": overlay,
        "boxes": boxes,
    }


In [11]:
def preview_detection_tool(
    resize_factor=1.0,

    do_bg_normalization=True,
    bg_blur_ksize=81,

    do_clahe=True,
    clahe_clip=2.0,
    clahe_grid=8,

    denoise_h=6,
    median_ksize=3,
    sharpen_amount=0.3,
    sharpen_sigma=1.0,

    threshold_method="adaptive_gaussian",
    block_size=41,
    c_value=9,
    invert_binary=False,

    morph_open=0,
    morph_close=3,
    dilate_iter=0,

    min_area=30,
    max_area=50000,
    min_width=3,
    min_height=3,
    max_width_ratio=0.95,
    max_height_ratio=0.95,
):
    global CURRENT_RESULTS, CURRENT_PARAMS

    CURRENT_PARAMS = {
        "resize_factor": resize_factor,
        "do_bg_normalization": do_bg_normalization,
        "bg_blur_ksize": bg_blur_ksize,
        "do_clahe": do_clahe,
        "clahe_clip": clahe_clip,
        "clahe_grid": clahe_grid,
        "denoise_h": denoise_h,
        "median_ksize": median_ksize,
        "sharpen_amount": sharpen_amount,
        "sharpen_sigma": sharpen_sigma,
        "threshold_method": threshold_method,
        "block_size": block_size,
        "c_value": c_value,
        "invert_binary": invert_binary,
        "morph_open": morph_open,
        "morph_close": morph_close,
        "dilate_iter": dilate_iter,
        "min_area": min_area,
        "max_area": max_area,
        "min_width": min_width,
        "min_height": min_height,
        "max_width_ratio": max_width_ratio,
        "max_height_ratio": max_height_ratio,
    }

    CURRENT_RESULTS = preprocess_for_detection(
        SOURCE_BGR,
        resize_factor=resize_factor,
        do_bg_normalization=do_bg_normalization,
        bg_blur_ksize=bg_blur_ksize,
        do_clahe=do_clahe,
        clahe_clip=clahe_clip,
        clahe_grid=clahe_grid,
        denoise_h=denoise_h,
        median_ksize=median_ksize,
        sharpen_amount=sharpen_amount,
        sharpen_sigma=sharpen_sigma,
        threshold_method=threshold_method,
        block_size=block_size,
        c_value=c_value,
        invert_binary=invert_binary,
        morph_open=morph_open,
        morph_close=morph_close,
        dilate_iter=dilate_iter,
        min_area=min_area,
        max_area=max_area,
        min_width=min_width,
        min_height=min_height,
        max_width_ratio=max_width_ratio,
        max_height_ratio=max_height_ratio,
    )

    res = CURRENT_RESULTS

    fig, axes = plt.subplots(2, 3, figsize=(20, 11))

    axes[0, 0].imshow(res["original_rgb"])
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(res["bg_norm"], cmap="gray")
    axes[0, 1].set_title("Background-normalized")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(res["enhanced"], cmap="gray")
    axes[0, 2].set_title("CLAHE / enhanced")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(res["processed"], cmap="gray")
    axes[1, 0].set_title("Processed for detection")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(res["contour_src"], cmap="gray")
    axes[1, 1].set_title("Foreground for contours")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(res["overlay"])
    axes[1, 2].set_title(f"Candidate boxes: {len(res['boxes'])}")
    axes[1, 2].axis("off")

    plt.tight_layout()
    plt.show()

In [12]:
interact(
    preview_detection_tool,

    resize_factor=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description="Resize"),

    do_bg_normalization=Checkbox(value=True, description="BG norm"),
    bg_blur_ksize=IntSlider(value=81, min=9, max=301, step=2, description="BG blur"),

    do_clahe=Checkbox(value=True, description="CLAHE"),
    clahe_clip=FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1, description="CLAHE clip"),
    clahe_grid=IntSlider(value=8, min=2, max=32, step=1, description="CLAHE grid"),

    denoise_h=IntSlider(value=6, min=0, max=25, step=1, description="Denoise"),
    median_ksize=IntSlider(value=3, min=0, max=15, step=1, description="Median"),

    sharpen_amount=FloatSlider(value=0.3, min=0.0, max=2.0, step=0.1, description="Sharpen"),
    sharpen_sigma=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Sharp σ"),

    threshold_method=Dropdown(
        options=["adaptive_gaussian", "adaptive_mean", "otsu", "none"],
        value="adaptive_gaussian",
        description="Thresh"
    ),
    block_size=IntSlider(value=41, min=3, max=151, step=2, description="Block"),
    c_value=IntSlider(value=9, min=-20, max=20, step=1, description="C"),
    invert_binary=Checkbox(value=False, description="Invert"),

    morph_open=IntSlider(value=0, min=0, max=11, step=1, description="Open"),
    morph_close=IntSlider(value=3, min=0, max=11, step=1, description="Close"),
    dilate_iter=IntSlider(value=0, min=0, max=5, step=1, description="Dilate"),

    min_area=IntSlider(value=30, min=1, max=5000, step=5, description="Min area"),
    max_area=IntSlider(value=50000, min=100, max=300000, step=100, description="Max area"),
    min_width=IntSlider(value=3, min=1, max=100, step=1, description="Min width"),
    min_height=IntSlider(value=3, min=1, max=100, step=1, description="Min height"),
    max_width_ratio=FloatSlider(value=0.95, min=0.05, max=1.0, step=0.01, description="Max W%"),
    max_height_ratio=FloatSlider(value=0.95, min=0.05, max=1.0, step=0.01, description="Max H%"),
);

interactive(children=(FloatSlider(value=1.0, description='Resize', max=2.0, min=0.5, step=0.05), Checkbox(valu…

In [8]:
save_current_outputs(prefix="kaithi_best", folder="opencv_ocr_outputs")

Saved outputs to: /home/azureuser/kaithi/OCD/opencv_ocr_outputs


Absolutely. Here is the same solution in a cleaner, properly structured format.

# Goal

You are **not** optimizing for final OCR text quality right now.

You are optimizing preprocessing so that your **OCD / text detection model** gets the **best bounding boxes**.

That means the preprocessing should:

* preserve text regions
* preserve layout
* reduce background noise
* avoid deleting faint strokes
* avoid overly aggressive binarization
* avoid removing structure too early

---

# Important difference

## For OCR recognition

Usually you want:

* very clean text
* line removal
* strong binarization
* text-only output

## For OCD / bounding box detection

Usually you want:

* layout-preserving image
* mild enhancement
* connected text blobs
* less aggressive cleanup

So for your current task, your idea is correct:
**background-normalized or lightly enhanced images are often better than OCR-ready binary images.**

---

# What this tool does

This OpenCV notebook tool lets you tune:

## Image enhancement

* background normalization
* CLAHE
* denoising
* sharpening

## Detection preparation

* thresholding
* morphology
* dilation

## Bounding-box filtering

* minimum area
* maximum area
* minimum width/height
* max width/height ratio

## Visual outputs

It shows:

* original image
* normalized image
* enhanced image
* processed image for detection
* contour source
* final overlay with candidate boxes

---

# Notebook code

## Cell 1 — Imports and setup

```python
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    interact,
    IntSlider,
    FloatSlider,
    Checkbox,
    Dropdown,
)

imgpath = r"D:\Work\multililpi\kaithi\potential-invention\image.png"

SOURCE_BGR = cv2.imread(imgpath)
if SOURCE_BGR is None:
    raise FileNotFoundError(f"Could not read image: {imgpath}")

CURRENT_RESULTS = {}
CURRENT_PARAMS = {}


def odd(x: int) -> int:
    x = int(x)
    return x if x % 2 == 1 else x + 1
```

---

## Cell 2 — Preprocessing + box generation

```python
def preprocess_for_detection(
    source_bgr,
    resize_factor=1.0,

    # grayscale / normalization
    do_bg_normalization=True,
    bg_blur_ksize=81,

    # contrast
    do_clahe=True,
    clahe_clip=2.0,
    clahe_grid=8,

    # denoise / sharpen
    denoise_h=6,
    median_ksize=3,
    sharpen_amount=0.3,
    sharpen_sigma=1.0,

    # thresholding
    threshold_method="adaptive_gaussian",   # adaptive_gaussian / adaptive_mean / otsu / none
    block_size=41,
    c_value=9,
    invert_binary=False,

    # morphology
    morph_open=0,
    morph_close=3,
    dilate_iter=0,

    # bbox filtering
    min_area=30,
    max_area=50000,
    min_width=3,
    min_height=3,
    max_width_ratio=0.95,
    max_height_ratio=0.95,
):
    img = source_bgr.copy()

    if resize_factor != 1.0:
        new_w = max(1, int(img.shape[1] * resize_factor))
        new_h = max(1, int(img.shape[0] * resize_factor))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    work = gray.copy()

    # 1) background normalization
    bg_norm = work.copy()
    if do_bg_normalization:
        k = odd(bg_blur_ksize)
        bg = cv2.GaussianBlur(work, (k, k), 0)
        bg_norm = cv2.divide(work, bg, scale=255)
        work = bg_norm.copy()

    # 2) local contrast
    enhanced = work.copy()
    if do_clahe:
        clahe = cv2.createCLAHE(
            clipLimit=float(clahe_clip),
            tileGridSize=(int(clahe_grid), int(clahe_grid)),
        )
        enhanced = clahe.apply(work)
        work = enhanced.copy()

    # 3) denoise
    denoised = work.copy()
    if denoise_h > 0:
        denoised = cv2.fastNlMeansDenoising(
            work, None, h=int(denoise_h), templateWindowSize=7, searchWindowSize=21
        )
        work = denoised.copy()

    # 4) median blur
    medianed = work.copy()
    if median_ksize >= 3:
        mk = odd(median_ksize)
        medianed = cv2.medianBlur(work, mk)
        work = medianed.copy()

    # 5) mild sharpen
    sharpened = work.copy()
    if sharpen_amount > 0:
        blur = cv2.GaussianBlur(work, (0, 0), sharpen_sigma)
        sharpened = cv2.addWeighted(work, 1.0 + sharpen_amount, blur, -sharpen_amount, 0)
        work = sharpened.copy()

    # 6) threshold
    if threshold_method == "none":
        binary = work.copy()
    elif threshold_method == "adaptive_gaussian":
        binary = cv2.adaptiveThreshold(
            work, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            odd(block_size),
            int(c_value),
        )
    elif threshold_method == "adaptive_mean":
        binary = cv2.adaptiveThreshold(
            work, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            odd(block_size),
            int(c_value),
        )
    elif threshold_method == "otsu":
        _, binary = cv2.threshold(work, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        raise ValueError("Unknown threshold_method")

    if threshold_method != "none" and invert_binary:
        binary = 255 - binary

    # 7) morphology
    processed = binary.copy()

    if threshold_method != "none":
        if morph_open > 0:
            ok = odd(morph_open)
            kernel = np.ones((ok, ok), np.uint8)
            processed = cv2.morphologyEx(processed, cv2.MORPH_OPEN, kernel)

        if morph_close > 0:
            ck = odd(morph_close)
            kernel = np.ones((ck, ck), np.uint8)
            processed = cv2.morphologyEx(processed, cv2.MORPH_CLOSE, kernel)

        if dilate_iter > 0:
            kernel = np.ones((3, 3), np.uint8)
            processed = cv2.dilate(processed, kernel, iterations=int(dilate_iter))

    # 8) contour source
    if threshold_method == "none":
        _, contour_src = cv2.threshold(work, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        contour_src = 255 - processed if np.mean(processed) > 127 else processed.copy()

    # 9) contours to boxes
    contours, _ = cv2.findContours(contour_src, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h_img, w_img = contour_src.shape[:2]
    max_w = int(w_img * max_width_ratio)
    max_h = int(h_img * max_height_ratio)

    boxes = []
    overlay = rgb.copy()

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h

        if area < min_area or area > max_area:
            continue
        if w < min_width or h < min_height:
            continue
        if w > max_w or h > max_h:
            continue

        boxes.append((x, y, w, h))
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (255, 0, 0), 2)

    return {
        "original_rgb": rgb,
        "gray": gray,
        "bg_norm": bg_norm,
        "enhanced": enhanced,
        "denoised": denoised,
        "sharpened": sharpened,
        "binary": binary,
        "processed": processed,
        "contour_src": contour_src,
        "overlay": overlay,
        "boxes": boxes,
    }
```

---

## Cell 3 — Preview function

```python
def preview_detection_tool(
    resize_factor=1.0,

    do_bg_normalization=True,
    bg_blur_ksize=81,

    do_clahe=True,
    clahe_clip=2.0,
    clahe_grid=8,

    denoise_h=6,
    median_ksize=3,
    sharpen_amount=0.3,
    sharpen_sigma=1.0,

    threshold_method="adaptive_gaussian",
    block_size=41,
    c_value=9,
    invert_binary=False,

    morph_open=0,
    morph_close=3,
    dilate_iter=0,

    min_area=30,
    max_area=50000,
    min_width=3,
    min_height=3,
    max_width_ratio=0.95,
    max_height_ratio=0.95,
):
    global CURRENT_RESULTS, CURRENT_PARAMS

    CURRENT_PARAMS = {
        "resize_factor": resize_factor,
        "do_bg_normalization": do_bg_normalization,
        "bg_blur_ksize": bg_blur_ksize,
        "do_clahe": do_clahe,
        "clahe_clip": clahe_clip,
        "clahe_grid": clahe_grid,
        "denoise_h": denoise_h,
        "median_ksize": median_ksize,
        "sharpen_amount": sharpen_amount,
        "sharpen_sigma": sharpen_sigma,
        "threshold_method": threshold_method,
        "block_size": block_size,
        "c_value": c_value,
        "invert_binary": invert_binary,
        "morph_open": morph_open,
        "morph_close": morph_close,
        "dilate_iter": dilate_iter,
        "min_area": min_area,
        "max_area": max_area,
        "min_width": min_width,
        "min_height": min_height,
        "max_width_ratio": max_width_ratio,
        "max_height_ratio": max_height_ratio,
    }

    CURRENT_RESULTS = preprocess_for_detection(
        SOURCE_BGR,
        resize_factor=resize_factor,
        do_bg_normalization=do_bg_normalization,
        bg_blur_ksize=bg_blur_ksize,
        do_clahe=do_clahe,
        clahe_clip=clahe_clip,
        clahe_grid=clahe_grid,
        denoise_h=denoise_h,
        median_ksize=median_ksize,
        sharpen_amount=sharpen_amount,
        sharpen_sigma=sharpen_sigma,
        threshold_method=threshold_method,
        block_size=block_size,
        c_value=c_value,
        invert_binary=invert_binary,
        morph_open=morph_open,
        morph_close=morph_close,
        dilate_iter=dilate_iter,
        min_area=min_area,
        max_area=max_area,
        min_width=min_width,
        min_height=min_height,
        max_width_ratio=max_width_ratio,
        max_height_ratio=max_height_ratio,
    )

    res = CURRENT_RESULTS

    fig, axes = plt.subplots(2, 3, figsize=(20, 11))

    axes[0, 0].imshow(res["original_rgb"])
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(res["bg_norm"], cmap="gray")
    axes[0, 1].set_title("Background-normalized")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(res["enhanced"], cmap="gray")
    axes[0, 2].set_title("CLAHE / enhanced")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(res["processed"], cmap="gray")
    axes[1, 0].set_title("Processed for detection")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(res["contour_src"], cmap="gray")
    axes[1, 1].set_title("Foreground for contours")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(res["overlay"])
    axes[1, 2].set_title(f"Candidate boxes: {len(res['boxes'])}")
    axes[1, 2].axis("off")

    plt.tight_layout()
    plt.show()
```

---

## Cell 4 — Interactive sliders

```python
interact(
    preview_detection_tool,

    resize_factor=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description="Resize"),

    do_bg_normalization=Checkbox(value=True, description="BG norm"),
    bg_blur_ksize=IntSlider(value=81, min=9, max=301, step=2, description="BG blur"),

    do_clahe=Checkbox(value=True, description="CLAHE"),
    clahe_clip=FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1, description="CLAHE clip"),
    clahe_grid=IntSlider(value=8, min=2, max=32, step=1, description="CLAHE grid"),

    denoise_h=IntSlider(value=6, min=0, max=25, step=1, description="Denoise"),
    median_ksize=IntSlider(value=3, min=0, max=15, step=1, description="Median"),

    sharpen_amount=FloatSlider(value=0.3, min=0.0, max=2.0, step=0.1, description="Sharpen"),
    sharpen_sigma=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Sharp σ"),

    threshold_method=Dropdown(
        options=["adaptive_gaussian", "adaptive_mean", "otsu", "none"],
        value="adaptive_gaussian",
        description="Thresh"
    ),
    block_size=IntSlider(value=41, min=3, max=151, step=2, description="Block"),
    c_value=IntSlider(value=9, min=-20, max=20, step=1, description="C"),
    invert_binary=Checkbox(value=False, description="Invert"),

    morph_open=IntSlider(value=0, min=0, max=11, step=1, description="Open"),
    morph_close=IntSlider(value=3, min=0, max=11, step=1, description="Close"),
    dilate_iter=IntSlider(value=0, min=0, max=5, step=1, description="Dilate"),

    min_area=IntSlider(value=30, min=1, max=5000, step=5, description="Min area"),
    max_area=IntSlider(value=50000, min=100, max=300000, step=100, description="Max area"),
    min_width=IntSlider(value=3, min=1, max=100, step=1, description="Min width"),
    min_height=IntSlider(value=3, min=1, max=100, step=1, description="Min height"),
    max_width_ratio=FloatSlider(value=0.95, min=0.05, max=1.0, step=0.01, description="Max W%"),
    max_height_ratio=FloatSlider(value=0.95, min=0.05, max=1.0, step=0.01, description="Max H%"),
);
```

---

# Recommended starting settings for your document

Use these first:

```python
resize_factor = 1.0

do_bg_normalization = True
bg_blur_ksize = 81

do_clahe = True
clahe_clip = 1.8
clahe_grid = 8

denoise_h = 4
median_ksize = 3
sharpen_amount = 0.2
sharpen_sigma = 1.0

threshold_method = "adaptive_gaussian"
block_size = 41
c_value = 7

morph_open = 0
morph_close = 3
dilate_iter = 0

min_area = 30
max_area = 50000
min_width = 3
min_height = 3
max_width_ratio = 0.95
max_height_ratio = 0.95
```

---

# How to tune the parameters

## 1. If boxes are too fragmented

This means one word/line is getting split into many tiny boxes.

Try:

* increase `morph_close`
* set `dilate_iter = 1`
* reduce threshold aggressiveness
* reduce denoising if faint strokes are disappearing

Best knobs:

* `morph_close`
* `dilate_iter`
* `block_size`
* `c_value`

---

## 2. If boxes are too merged

This means multiple words/regions are becoming one big box.

Try:

* reduce `morph_close`
* set `dilate_iter = 0`
* reduce CLAHE clip
* use milder thresholding

Best knobs:

* `morph_close`
* `dilate_iter`
* `clahe_clip`

---

## 3. If faint text is missing

This means the detector is ignoring weak handwriting.

Try:

* lower `denoise_h`
* reduce sharpening
* try `threshold_method = "none"`
* try larger `block_size`

Best knobs:

* `denoise_h`
* `sharpen_amount`
* `threshold_method`
* `block_size`

---

## 4. If stains/noise are getting boxes

This means background dirt is being treated as text.

Try:

* slightly increase denoising
* increase `min_area`
* use a small `morph_open`
* reduce CLAHE if it is amplifying noise

Best knobs:

* `denoise_h`
* `min_area`
* `morph_open`
* `clahe_clip`

---

# What each major parameter does

## `bg_blur_ksize`

Controls background normalization strength.

* larger value = stronger illumination correction
* useful for old uneven scans

For your type of page:

* try `61–121`

---

## `clahe_clip`

Controls local contrast enhancement.

* low = safer, less noise
* high = stronger text contrast but also more noise

Good range:

* `1.5–2.5`

---

## `denoise_h`

Controls denoising strength.

* low = preserves faint text
* high = removes more stains/noise, but can erase handwriting

Good range:

* `3–8`

---

## `sharpen_amount`

Adds mild edge enhancement.

* too much will create broken noisy outlines
* for detection, keep this low

Good range:

* `0.0–0.4`

---

## `threshold_method`

Options:

* `adaptive_gaussian`
* `adaptive_mean`
* `otsu`
* `none`

For your case, try in this order:

1. `adaptive_gaussian`
2. `adaptive_mean`
3. `none`

`none` can sometimes work surprisingly well for detection pipelines.

---

## `morph_close`

Very important for bounding boxes.

* low = separate fragments remain separate
* high = nearby text joins together

Good starting range:

* `1–5`

---

## `dilate_iter`

Expands white foreground regions.

* helps faint disconnected strokes merge
* too much merges unrelated text

Usually:

* `0` or `1`

---

# Practical strategy

## Best workflow for you

### Branch A — Detection input

Use this for OCD / box prediction:

* background normalization
* mild CLAHE
* light denoise
* mild threshold or no threshold
* mild morphology
* no line removal

### Branch B — Recognition input

Use this later for OCR on cropped boxes:

* stronger cleanup
* maybe line removal
* maybe separate OCR preprocessing per crop

This is usually much stronger than using one single image for both tasks.

---

# My recommendation based on your screenshot

For **bounding boxes**, start by focusing mostly on these preview panels:

* **Background-normalized**
* **CLAHE / enhanced**
* **Processed for detection**
* **Overlay**

Do not try to make the image look “perfect for reading.”
Try to make it look:

* structurally stable
* low-noise
* with connected text regions

That is what helps detection most.

---

# Next improvement I recommend

The next useful version of this tool would show:

* detected boxes on image
* cropped candidate regions in a grid
* box count statistics
* maybe width/height histograms

That would make tuning much easier because you could directly see whether the model is splitting or merging text badly.
